# House Price Prediction

Week 9 machine-learning project. Follow the seven days in order. The dataset is in the same folder as this notebook.

## Day 1 — Understand the data
Load the CSV, inspect the columns, missing values, and relationships.

In [ ]:
from pathlib import Path
import pandas as pd
import matplotlib.pyplot as plt
ROOT=Path.cwd()
if not (ROOT/"house_data.csv").exists(): ROOT=ROOT.parent
df=pd.read_csv(ROOT/"house_data.csv")
print(df.shape); display(df.head()); display(df.isna().sum()); display(df.describe(include="all").T)


In [ ]:
plt.scatter(df.Area,df.Price,alpha=.6); plt.xlabel("Area"); plt.ylabel("Price"); plt.title("Area vs price"); plt.show()
display(df.groupby("Location").Price.median().sort_values(ascending=False))


## Day 2 — Prepare data and split
The test set is held aside. Imputation, scaling, and category encoding happen inside the model pipeline, fitted on training data only.

In [ ]:
import numpy as np
from sklearn.model_selection import train_test_split
y=pd.to_numeric(df.Price,errors="coerce")
X=df.drop(columns=["Price"] + (["Property_ID"] if "Property_ID" in df else []))
for col in X:
 v=pd.to_numeric(X[col],errors="coerce")
 if v.notna().sum() >= .8*len(X): X[col]=v
valid=y.notna(); X=X.loc[valid]; y=y.loc[valid]
X_train,X_test,y_train,y_test=train_test_split(X,y,test_size=.2,random_state=42)
print("Train rows:",len(X_train),"Test rows:",len(X_test))


## Day 3 — Linear regression from scratch
For one input x, slope = sum((x−mean(x))(y−mean(y))) / sum((x−mean(x))²), and intercept = mean(y) − slope × mean(x). This introduces the idea; the evaluated model uses scikit-learn with multiple features.

In [ ]:
x=df.Area.astype(float).to_numpy(); yy=df.Price.astype(float).to_numpy()
slope=((x-x.mean())*(yy-yy.mean())).sum()/((x-x.mean())**2).sum()
intercept=yy.mean()-slope*x.mean()
print(f"Price ≈ {intercept:,.0f} + {slope:,.0f} × Area")


## Day 4 — Train scikit-learn models
Compare a linear regression baseline and a random forest using the same train/test split.

In [ ]:
from sklearn.compose import ColumnTransformer
from sklearn.preprocessing import OneHotEncoder,StandardScaler
from sklearn.impute import SimpleImputer
from sklearn.pipeline import Pipeline
from sklearn.linear_model import LinearRegression
from sklearn.ensemble import RandomForestRegressor
from sklearn.metrics import mean_absolute_error,mean_squared_error,r2_score
num=X_train.select_dtypes(include="number").columns.tolist(); cat=[c for c in X_train if c not in num]
prep=ColumnTransformer([("num",Pipeline([('imputer',SimpleImputer(strategy='median')),('scale',StandardScaler())]),num),("cat",Pipeline([('imputer',SimpleImputer(strategy='most_frequent')),('onehot',OneHotEncoder(handle_unknown='ignore'))]),cat)])
models={"Linear Regression":Pipeline([('prep',prep),('model',LinearRegression())]),"Random Forest":Pipeline([('prep',prep),('model',RandomForestRegressor(n_estimators=400,min_samples_leaf=2,random_state=42))])}
results={}
for name,model in models.items():
 model.fit(X_train,y_train); results[name]=model.predict(X_test)
 print(name,"trained")


## Day 5 — Evaluate and visualize
Compare at least three metrics, then inspect actual versus predicted values.

In [ ]:
for name,prediction in results.items():
 print(name,"MAE",mean_absolute_error(y_test,prediction),"MSE",mean_squared_error(y_test,prediction),"RMSE",mean_squared_error(y_test,prediction)**.5,"R2",r2_score(y_test,prediction))
best=min(results,key=lambda name:mean_squared_error(y_test,results[name])**.5); pred=results[best]
plt.scatter(y_test,pred); lo=min(y_test.min(),pred.min()); hi=max(y_test.max(),pred.max()); plt.plot([lo,hi],[lo,hi],"--",color="red"); plt.xlabel("Actual price"); plt.ylabel("Predicted price"); plt.title("Actual vs predicted — "+best); plt.show()


## Day 6 — Check features and model errors
Permutation importance estimates how much test-set error changes when one feature is shuffled. It is not a causal explanation.

In [ ]:
from sklearn.inspection import permutation_importance
pi=permutation_importance(models[best],X_test,y_test,n_repeats=20,random_state=42,scoring="neg_mean_absolute_error")
display(pd.DataFrame({"feature":X.columns,"importance":pi.importances_mean}).sort_values("importance",ascending=False))
residual=y_test.to_numpy()-pred
plt.scatter(pred,residual); plt.axhline(0,color="red",ls="--"); plt.xlabel("Predicted price"); plt.ylabel("Actual − predicted"); plt.title("Residual check"); plt.show()


## Day 7 — Interpret and present
Review `model_evaluation_report.md` and `predictions_vs_actual.png`. The dataset has 300 properties and does not provide currency or sale dates. Treat the results as a practice model, not a real-world appraisal.